# Session 9 — Advanced Retrieval (Kids Science PDF)

This notebook evaluates multiple retriever strategies (Naive, BM25, Rerank, Multi-Query, Parent-Doc, Ensemble) on the Kids Science PDF dataset using RAGAS (v0.2.10), following Session 7 structure and dependencies.


## Cell 1 — Pre-flight and environment (Session 7 style)
- Set LangSmith tracing and project
- Provide API keys via prompts
- Download NLTK resources


In [1]:
import os
import getpass
import nltk

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "s9-advanced-retrieval"

# Prompt for keys if not already set in env
if not os.environ.get("LANGCHAIN_API_KEY"):
    os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")
# Optional Cohere for reranking
if not os.environ.get("COHERE_API_KEY"):
    cohere_key = getpass.getpass("Cohere API Key (for reranker, optional):")
    if cohere_key:
        os.environ["COHERE_API_KEY"] = cohere_key

# NLTK downloads (Session 7)
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')


[nltk_data] Downloading package punkt to /home/lalit/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/lalit/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

## Cell 2 — Imports, config, and small helpers
- Session 7 compatible imports only
- Simple latency timer


In [2]:
import time
from typing import Dict, Any, List

from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Qdrant

# Helper for latency timing
class LatencyTimer:
    def __enter__(self):
        self.start = time.time()
        return self
    def __exit__(self, exc_type, exc, tb):
        self.end = time.time()
        self.elapsed_s = self.end - self.start

# Models (confirmed)
GENERATOR_LLM = "gpt-4.1-nano"
RAG_LLM = "gpt-4.1-mini"
EVAL_LLM = "gpt-4.1"
EMBED_MODEL = "text-embedding-3-small"


## Cell 3 — Load Kids Science PDFs (data/grade3/)
- Use DirectoryLoader + PyMuPDFLoader
- Keep cleaning minimal


In [ ]:
from pathlib import Path

DATA_DIR = Path("data/grade3")
assert DATA_DIR.exists(), f"Missing dataset folder: {DATA_DIR}"

loader = DirectoryLoader(str(DATA_DIR), glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()
print(f"Loaded {len(docs)} documents from {DATA_DIR}")
print("Sample metadata:", docs[0].metadata if docs else "<no docs>")


Loaded 25 documents from data/grade3
Sample metadata: {'producer': 'WeasyPrint 65.1', 'creator': 'ChatGPT', 'creationdate': '', 'source': 'data/grade3/Grade 3 Science Booklets (Ontario Curriculum Aligned).pdf', 'file_path': 'data/grade3/Grade 3 Science Booklets (Ontario Curriculum Aligned).pdf', 'total_pages': 25, 'format': 'PDF 1.7', 'title': 'Grade 3 Science Booklets (Ontario Curriculum Aligned)', 'author': 'ChatGPT Deep Research', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}


## Cell 4 — Split into child/parent chunks (Session 7 style)
- RecursiveCharacterTextSplitter 500/50
- Persist parent metadata for Parent-Doc later


In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

parent_docs = docs  # keep a handle for Parent-Doc
child_docs = text_splitter.split_documents(docs)
print(f"Parent docs: {len(parent_docs)} | Child chunks: {len(child_docs)}")


Parent docs: 25 | Child chunks: 186


## Cell 5 — Base embeddings + single Qdrant index (in-memory)
- text-embedding-3-small (Session 7)
- One Qdrant collection reused across variants
- Base retriever with k=8


In [5]:
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

vectorstore = Qdrant.from_documents(
    documents=child_docs,
    embedding=embeddings,
    location=":memory:",  # same as Session 7
    collection_name="kids_science_s9",
)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 8})
print("Qdrant collection ready; base retriever configured (k=8)")


Qdrant collection ready; base retriever configured (k=8)
